In [ ]:
from dataclasses import dataclass

import numpy as np
import jax.numpy as jnp


@dataclass(frozen=True)
class TestCurve:
    name: str
    curve_type: str
    x: object
    values: object


@dataclass(frozen=True)
class TestParameter:
    name: str
    curves: tuple

    @property
    def curve_types(self):
        return tuple(curve.curve_type for curve in self.curves)

    def get(self, curve_type):
        for curve in self.curves:
            if curve.curve_type == curve_type:
                return curve
        raise KeyError(f"Unknown curve type {curve_type!r}. Available: {self.curve_types}")

    def values(self, curve_type):
        return self.get(curve_type).values

    def __getitem__(self, key):
        if isinstance(key, int):
            return self.curves[key]
        return self.get(key)

    def __iter__(self):
        return iter(self.curves)

    def __len__(self):
        return len(self.curves)


@dataclass(frozen=True)
class TestDataset:
    parameters: tuple

    @property
    def names(self):
        return tuple(parameter.name for parameter in self.parameters)

    @property
    def all_curves(self):
        return tuple(curve for parameter in self.parameters for curve in parameter.curves)

    def get(self, name):
        for parameter in self.parameters:
            if parameter.name == name:
                return parameter
        raise KeyError(f"Unknown parameter {name!r}. Available: {self.names}")

    def __getitem__(self, key):
        if isinstance(key, int):
            return self.parameters[key]
        return self.get(key)

    def __iter__(self):
        return iter(self.parameters)

    def __len__(self):
        return len(self.parameters)


NONZERO_EPS = 1.0e-6


def _replace_zeros(values, replacement=NONZERO_EPS):
    values = jnp.asarray(values)
    replacement = jnp.asarray(replacement, dtype=values.dtype)
    return jnp.where(values == 0, replacement, values)


def _prepare_curve_values(values, positive=True):
    if positive:
        values = jnp.abs(values)
    return _replace_zeros(values)


def _curve_templates(length, positive=True):
    x = _replace_zeros(jnp.linspace(0.0, 1.0, length, dtype=jnp.float32))
    center = 0.5
    width = 0.12
    delta = jnp.full(length, NONZERO_EPS, dtype=jnp.float32).at[length // 2].set(1.0)

    curves = {
        "constant": jnp.ones_like(x),
        "zero": jnp.full_like(x, NONZERO_EPS),
        "identity function": x,
        "linear": 2.0 * x - 1.0,
        "quadratic": x**2,
        "power law": jnp.sqrt(x),
        "gaussian": jnp.exp(-0.5 * ((x - center) / width) ** 2),
        "delta": delta,
        "square pulse": jnp.where((x >= 0.35) & (x <= 0.65), 1.0, 0.0),
        "smooth step": 0.5 * (1.0 + jnp.tanh((x - center) / 0.08)),
        "sinusoidal": 0.5 * (1.0 - jnp.cos(2.0 * jnp.pi * x)),
        "lorentzian": 1.0 / (1.0 + ((x - center) / width) ** 2),
        "exponential decay": jnp.exp(-4.0 * x),
    }
    curves = {name: _prepare_curve_values(values, positive=positive) for name, values in curves.items()}
    return x, curves


def _with_complex_values(curves, x, positive=True):
    center = 0.5
    width = 0.12
    shifted_center = 0.62
    delta_imag = jnp.full(x.shape, NONZERO_EPS, dtype=jnp.float32).at[x.shape[0] // 3].set(1.0)
    imag_curves = {
        "constant": 0.5 * jnp.ones_like(x),
        "zero": jnp.full_like(x, 2.0 * NONZERO_EPS),
        "identity function": 1.0 - x,
        "linear": 0.75 - 1.5 * x,
        "quadratic": (1.0 - x)**2,
        "power law": jnp.sqrt(1.0 - x),
        "gaussian": jnp.exp(-0.5 * ((x - shifted_center) / width) ** 2),
        "delta": delta_imag,
        "square pulse": jnp.where((x >= 0.55) & (x <= 0.85), 1.0, NONZERO_EPS),
        "smooth step": 0.5 * (1.0 - jnp.tanh((x - center) / 0.08)),
        "sinusoidal": 0.5 * (1.0 + jnp.sin(2.0 * jnp.pi * x)),
        "lorentzian": 1.0 / (1.0 + ((x - shifted_center) / width) ** 2),
        "exponential decay": jnp.exp(-2.0 * x),
    }
    imag_curves = {name: _prepare_curve_values(values, positive=positive) for name, values in imag_curves.items()}
    return {
        name: jnp.asarray(values, dtype=jnp.complex64) + 1j * jnp.asarray(imag_curves[name], dtype=jnp.complex64)
        for name, values in curves.items()
    }


def _normalise_parameter_names(parameter_names):
    if isinstance(parameter_names, str):
        names = (parameter_names,)
    else:
        names = tuple(parameter_names)

    if not names:
        raise ValueError("At least one parameter name is required.")

    return tuple(str(name) for name in names)


def make_test_dataset(parameter_names, length=100, seed=None, curve_types=None, complex_values=False, positive=True):
    names = _normalise_parameter_names(parameter_names)
    length = int(length)
    if length <= 0:
        raise ValueError("length must be positive.")

    x, available_curves = _curve_templates(length, positive=positive)
    if complex_values:
        available_curves = _with_complex_values(available_curves, x, positive=positive)

    if curve_types is None:
        selected_curve_types = tuple(available_curves)
    else:
        selected_curve_types = tuple(curve_types)
        unknown = set(selected_curve_types) - set(available_curves)
        if unknown:
            raise ValueError(f"Unknown curve type(s): {sorted(unknown)}")

    rng = np.random.default_rng(seed)
    parameters = []
    for name in names:
        shuffled_curve_types = tuple(rng.permutation(selected_curve_types))
        curves = tuple(
            TestCurve(
                name=name,
                curve_type=curve_type,
                x=x,
                values=available_curves[curve_type],
            )
            for curve_type in shuffled_curve_types
        )
        parameters.append(TestParameter(name=name, curves=curves))

    return TestDataset(parameters=tuple(parameters))


def make_test_parameter(name, length=100, seed=None, curve_types=None, complex_values=False, positive=True):
    return make_test_dataset(
        name,
        length=length,
        seed=seed,
        curve_types=curve_types,
        complex_values=complex_values,
        positive=positive,
    )[name]


# Example use from another notebook:
# %run ./test.ipynb
# import math
# import numpy as np
# import matplotlib.pyplot as plt
#
# function_under_test = <function_that_takes_the_test_arrays>
#
# test_variable_names = ["<test_variable_name_1>", "<test_variable_name_2>"]
# array_length = 100
# random_seed = 0
# curve_types_to_test = None 
# complex_values = False
# positive = True
# x_axis_name = "Time"
# y_axis_name = "y-axis name"
# plot_columns = 3
# fixed_args_before_test_arrays = () 
# fixed_args_after_test_arrays = () 
#
#
# dataset = make_test_dataset(
#     test_variable_names,
#     length=array_length,
#     seed=random_seed,
#     curve_types=curve_types_to_test,
#     complex_values=complex_values,
#     positive=positive,
# )
# test_combinations = list(zip(*(dataset[name] for name in dataset.names)))
#
# results = []
# for curves in test_combinations:
#     test_arrays = [curve.values for curve in curves]
#     output_array = function_under_test(
#         *fixed_args_before_test_arrays,
#         *test_arrays,
#         *fixed_args_after_test_arrays,
#     )
#     results.append((curves, output_array))
#
# if not results:
#     raise ValueError("No test combinations were generated.")
#
# ncols = min(max(1, plot_columns), len(results))
# nrows = math.ceil(len(results) / ncols)
# fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows), squeeze=False)
# axes = axes.ravel()
#
# for ax, (curves, y_values) in zip(axes, results):
#     y_values = np.asarray(y_values)
#     time = range(len(y_values))
#     title = ", ".join(f"{curve.name}: {curve.curve_type}" for curve in curves)
#     if np.iscomplexobj(y_values):
#         y_values = np.abs(y_values)
#     ax.plot(time, y_values, marker="o", markersize=2, linewidth=1.5)
#     ax.set_title(title)
#     ax.set_xlabel(x_axis_name)
#     ax.set_ylabel(y_axis_name)
#
# for ax in axes[len(results):]:
#     ax.set_visible(False)
#
# fig.tight_layout()
